In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
file_path = "Global378+Tibet15+Fossil73_更正.xlsx"

# modern plants
df_modern = pd.read_excel(
    file_path,
    sheet_name="plants"
)

# fossil
df_fossil = pd.read_excel(
    file_path,
    sheet_name="fossils"
)

feature_cols = [
'Lobed','No Teeth','Teeth Regul','Teeth Close',
'Teeth Round','Teeth Acute','Teeth Compo',
'Nanophyl','Leptophyl','Leptophyl 2',
'Microphyl 1','Microphyl 2','Microphyl 3',
'Mesophyl 1','Mesophyl 2','Mesophyl 3',
'Apex Emargi','Apex Round','Apex Acute',
'Apex Attenu',
'Base Cordat','Base Round','Base Acute',
'L:W <1:1','L:W 1-2:1','L:W 2-3:1',
'L:W 3-4:1','L:W >4:1',
'Obovate','Elliptic','Ovate'
]

target_cols = [
'MAT',
'WMMT',
'CMMT',
'GROWSEAS',
'RH',
'SH',
'ENTHAL'
]

X = df_modern[feature_cols]

rkf = RepeatedKFold(
    n_splits=10,
    n_repeats=5,
    random_state=42
)

performance = {}

fossil_predictions = pd.DataFrame()

for target in target_cols:

    print(f"\nProcessing {target}")

    y = df_modern[target]

    r2_list = []
    rmse_list = []
    mae_list = []

    for train_idx, test_idx in rkf.split(X):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = AdaBoostRegressor(

            estimator=DecisionTreeRegressor(
                max_depth=4,
                random_state=42
            ),

            n_estimators=300,

            learning_rate=0.05,

            random_state=42
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(X_test)

        r2_list.append(
            r2_score(
                y_test,
                y_pred
            )
        )

        rmse_list.append(
            np.sqrt(
                mean_squared_error(
                    y_test,
                    y_pred
                )
            )
        )

        mae_list.append(
            mean_absolute_error(
                y_test,
                y_pred
            )
        )

    performance[target] = {

        "R2_mean": np.mean(r2_list),
        "R2_std": np.std(r2_list),

        "RMSE_mean": np.mean(rmse_list),
        "RMSE_std": np.std(rmse_list),

        "MAE_mean": np.mean(mae_list),
        "MAE_std": np.std(mae_list)

    }

    final_model = AdaBoostRegressor(

        estimator=DecisionTreeRegressor(
            max_depth=4,
            random_state=42
        ),

        n_estimators=300,

        learning_rate=0.05,

        random_state=42
    )

    final_model.fit(
        X,
        y
    )

    joblib.dump(
        final_model,
        f"{target}_AdaBoost.joblib"
    )
    X_fossil = df_fossil[feature_cols]

    fossil_predictions[target] = final_model.predict(
        X_fossil
    )
table1 = pd.DataFrame(performance).T

print(table1)

table1.to_excel(
    "AdaBoost_CV_Performance.xlsx"
)


Processing MAT

Processing WMMT

Processing CMMT

Processing GROWSEAS

Processing RH

Processing SH

Processing ENTHAL
           R2_mean    R2_std  RMSE_mean  RMSE_std  MAE_mean   MAE_std
MAT       0.825961  0.047348   3.010832  0.388322  2.375723  0.300043
WMMT      0.665025  0.076362   2.885384  0.293283  2.258712  0.260738
CMMT      0.770093  0.074215   5.021354  1.034667  3.751466  0.640422
GROWSEAS  0.796676  0.056023   1.429719  0.217922  1.066290  0.137464
RH        0.610716  0.093687   7.015217  0.681359  5.613420  0.585899
SH        0.772305  0.053123   1.603174  0.184491  1.248157  0.148668
ENTHAL    0.801222  0.046252   8.663409  0.923235  6.791144  0.730352
